# Task 3: Caching Experiment (Queries 1, 6)

- Caches a reduced `integrated_taxi_trips` view containing only the columns used by Queries 3 and 5.
- Compares Query 1 and Query 6 before and after caching.

## Connect to Spark and imports

In [1]:
import sys
from time import perf_counter

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-Caching-Queries1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')

QUERY_NAMES = (
    'query_01_monthly_zone_demand',
    'query_06_monthly_taxi_demand_trend',
)
REPETITIONS = 3

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 10:13:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Measure uncached execution

In [2]:
spark.catalog.clearCache()
register_integrated_taxi_trips(spark)

uncached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

print('Uncached measurements complete.')

26/09/19 10:13:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/19 10:13:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Uncached measurements complete.


In [3]:
for query_name in QUERY_NAMES:
    print(f'Uncached plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

Uncached plan: query_01_monthly_zone_demand
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_date#17, pickup_zone_name#24, date#58]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<pickup_date:string,pickup_zone_name:string>

(2) Project
Output [2]: [pickup_zone_name#24, substring(pickup_date#17, 1, 7) AS pickup_month#1412]
Input [3]: [pickup_date#17, pickup_zone_name#24, date#58]

(3) HashAggregate
Input [2]: [pickup_zone_name#24, pickup_month#1412]
Keys [2]: [pickup_zone_name#24, pickup_month#1412]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1543L]
Results [3]: [pickup_zone_name#24, pickup_month#1412, count#1544L]

(4) Exchange
Input [3]: [pickup_zone_name#24, pickup_month#1412, c

## Cache a reduced shared input

- Caching the full Gold table exceeds available executor memory.
- Queries 1 and 6 need only the below two columns.

In [4]:
CACHE_COLUMNS = ('pickup_date', 'pickup_zone_name')

cache_input = spark.table(INTEGRATED_TABLE_VIEW).select(*CACHE_COLUMNS)
cache_input.createOrReplaceTempView(INTEGRATED_TABLE_VIEW)

cache_started_at = perf_counter()
spark.catalog.cacheTable(INTEGRATED_TABLE_VIEW)
cached_input_rows = spark.table(INTEGRATED_TABLE_VIEW).count()
cache_materialization_seconds = perf_counter() - cache_started_at

assert spark.catalog.isCached(INTEGRATED_TABLE_VIEW)
print(f'Cached input rows: {cached_input_rows}')
print(f'Cache materialization: {cache_materialization_seconds:.3f} seconds')

Cached input rows: 9417833
Cache materialization: 1.492 seconds


## Measure cached execution and verify results

In [5]:
cached_results = {
    query_name: measure_registered_query(spark, query_name, REPETITIONS)
    for query_name in QUERY_NAMES
}

for query_name in QUERY_NAMES:
    assert rows_match(
        uncached_results[query_name]['rows'],
        cached_results[query_name]['rows'],
    ), (
        f'Results differ for {query_name}.'
    )

print('All cached results match the uncached results within float tolerance.')

All cached results match the uncached results within float tolerance.


In [6]:
for query_name in QUERY_NAMES:
    print(f'Cached plan: {query_name}')
    execute_registered_query(spark, query_name).explain('formatted')

Cached plan: query_01_monthly_zone_demand
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- HashAggregate (9)
         +- Exchange (8)
            +- HashAggregate (7)
               +- Project (6)
                  +- Scan In-memory table integrated_taxi_trips (1)
                        +- InMemoryRelation (2)
                              +- * Project (5)
                                 +- * ColumnarToRow (4)
                                    +- Scan parquet  (3)


(1) Scan In-memory table integrated_taxi_trips
Output [2]: [pickup_date#17, pickup_zone_name#24]
Arguments: [pickup_date#17, pickup_zone_name#24]

(2) InMemoryRelation
Arguments: [pickup_date#17, pickup_zone_name#24], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan parquet 
Output [3]: [pickup_date#17, pickup_zone_name#24, date#58]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trips]
ReadSchema: struct<pickup_date:string,pick

## Benchmark summary

In [7]:
summary = {}
for query_name in QUERY_NAMES:
    uncached_median = uncached_results[query_name]['median_seconds']
    cached_median = cached_results[query_name]['median_seconds']
    summary[query_name] = {
        'uncached_samples_seconds': [
            round(value, 3) for value in uncached_results[query_name]['samples_seconds']
        ],
        'uncached_median_seconds': round(uncached_median, 3),
        'cached_samples_seconds': [
            round(value, 3) for value in cached_results[query_name]['samples_seconds']
        ],
        'cached_median_seconds': round(cached_median, 3),
        'speedup_factor': round(uncached_median / cached_median, 2),
        'cache_materialization_seconds': round(cache_materialization_seconds, 3),
    }

summary

{'query_01_monthly_zone_demand': {'uncached_samples_seconds': [1.055,
   1.714,
   0.841],
  'uncached_median_seconds': 1.055,
  'cached_samples_seconds': [0.321, 0.309, 0.282],
  'cached_median_seconds': 0.309,
  'speedup_factor': 3.42,
  'cache_materialization_seconds': 1.492},
 'query_06_monthly_taxi_demand_trend': {'uncached_samples_seconds': [0.564,
   0.555,
   0.601],
  'uncached_median_seconds': 0.564,
  'cached_samples_seconds': [0.474, 0.311, 0.236],
  'cached_median_seconds': 0.311,
  'speedup_factor': 1.82,
  'cache_materialization_seconds': 1.492}}

In [8]:
spark.catalog.uncacheTable(INTEGRATED_TABLE_VIEW)
spark.catalog.clearCache()
spark.stop()